In [1]:
import fs from "node:fs"

In [2]:
const input = fs.readFileSync("input.txt", "utf-8")

In [3]:
const sample = `\
..@@.@@@@.
@@@.@.@.@@
@@@@@.@.@@
@.@@@@..@.
@@.@@@@.@@
.@@@@@@@.@
.@.@.@.@@@
@.@@@.@@@@
.@@@@@@@@.
@.@.@@@.@.\
`

The first part of today's problem is to count how many places on the grid have `@` but are not surrounded by `@`. Sounds like a convolution problem, but since we're in javascript we'll implement it ourselves. First we convert the input string to ones and zeros. 

In [4]:
const process = inp => inp.split('\n').map(r => r.split('').map(e => e === "@" ? 1 : 0))
process(sample).slice(0,2)

[
  [
    0, 0, 1, 1, 0,
    1, 1, 1, 1, 0
  ],
  [
    1, 1, 1, 0, 1,
    0, 1, 0, 1, 1
  ]
]

Let's also add a 2d map function to make our lives a bit easier.

In [5]:
const map2 = (a, lam) => a.map(r => r.map(lam))

let's also have a function to pretty print the arrays

In [6]:
a[0].join("")

ReferenceError: a is not defined

In [7]:
const pp = a => console.log(a.map(r => r.join(" ")).join("\n"))
pp(a)

ReferenceError: a is not defined

Now we can do something like this

In [8]:
pp(map2(process(sample), x => x + 1))

1 1 2 2 1 2 2 2 2 1
2 2 2 1 2 1 2 1 2 2
2 2 2 2 2 1 2 1 2 2
2 1 2 2 2 2 1 1 2 1
2 2 1 2 2 2 2 1 2 2
1 2 2 2 2 2 2 2 1 2
1 2 1 2 1 2 1 2 2 2
2 1 2 2 2 1 2 2 2 2
1 2 2 2 2 2 2 2 2 1
2 1 2 1 2 2 2 1 2 1


Hmm, so I figure I can implement a more general convolution function. For inputs we have
- a: the array
- ker: the kernel
- pad: the padding amount

In [9]:
const a = process(sample)

In [10]:
const ker = [[1, 1, 1], [1, 0, 1], [1, 1, 1]] // this counts neighbors without the middle one
pp(ker)

1 1 1
1 0 1
1 1 1


In term of indexing, we imagine hovering the kernel over the array, each time the kernel is centered in another element. We need to understand the offsets:
- if `ks=1` the offsets are (0)
- if `ks=3` the offsets are (-1, 0, 1)
- if `ks=5` the offsets are (-2, -1, 0, 1, 2)

So it looks like as a rule we can iterate on `range(-ks//2, ks//2 + 1)`

In [11]:
const hks = Math.floor(ker.length / 2)
hks

1

let's calculate the convolution around `a[1, 1]`. The chunk looks like
```
0 0 1
1 1 1
1 1 1
```
So the number of neighbors is 6.

In [12]:
const i = 1;
const j = 1;
let sum = 0;
for (let k = -hks; k <= hks; k++) {
  for (let l = -hks; l <= hks; l++) {
    // console.log(i, j, k, l)
    // console.log(a[i+k][j+l], ker[k+hks][l+hks])
    sum += a[i+k][j+l] * ker[k+hks][l+hks]
  }
}
sum

6

Cool. Now, we might be tempted to stick it in a loop and be done with it, but not so fast. We need to handle the edge cases where the array kernel goes out of the array's bounds. For our case, to simplify, let's just imagine the array is surrounded by 0. I can think of two approaches
- Check out of bounds in the loop
- Pad the input array

I think out of bounds will be simple enough.

In [13]:
let res = map2(a, x => 0) // init result array
for (let i = 0; i < a.length; i++) {
  for (let j = 0; j < a[0].length; j++) {
    for (let k = -hks; k <= hks; k++) {
      for (let l = -hks; l <= hks; l++) {
        if (i+k < 0 || i+k >= a.length  || j+l < 0 || j+l >= a[0].length) {
          continue
        }
        res[i][j] += a[i+k][j+l] * ker[k+hks][l+hks]
      }
    }
  }
}
pp(res)

2 4 3 3 3 3 3 4 3 3
3 6 6 7 4 6 4 7 5 4
4 7 6 7 5 6 2 5 4 4
4 7 6 7 7 6 4 5 4 5
3 5 7 7 8 7 5 5 4 3
4 4 6 5 7 6 6 5 7 4
3 4 7 6 7 5 7 6 7 4
2 5 6 6 6 6 6 7 7 4
3 5 5 7 6 7 6 7 5 4
1 4 3 5 4 5 4 5 2 2


Looks sane, let's put it in a function

In [14]:
function conv2d(a, ker) {
  if (ker.length != ker[0].length || ker.length % 2 == 0) {
    throw new Error("illegal kernel")
  }
  let res = map2(a, x => 0) // init result array
  for (let i = 0; i < a.length; i++) {
    for (let j = 0; j < a[0].length; j++) {
      for (let k = -hks; k <= hks; k++) {
        for (let l = -hks; l <= hks; l++) {
          if (i+k < 0 || i+k >= a.length  || j+l < 0 || j+l >= a[0].length) {
            continue
          }
          res[i][j] += a[i+k][j+l] * ker[k+hks][l+hks]
        }
      }
    }
  }
  return res;
}

Okay, now let's iterate on both arrays and sum how many times there's a 1 in `a` and less than 4 in `res`

In [15]:
res = conv2d(a, ker);
let sum = 0;
for (let i = 0; i < a.length; i++) {
  for (let j = 0; j < a[0].length; j++) {
    if (a[i][j] === 1 && res[i][j] < 4) {
      sum += 1
    }
  }
}
sum

13

Correct according to the question, let's put it in a function and try the input

In [16]:
function part1(input) {
  const a = process(input)
  const ker = [[1, 1, 1], [1, 0, 1], [1, 1, 1]] // this counts neighbors without the middle one
  const res = conv2d(a, ker)
  let sum = 0;
  for (let i = 0; i < a.length; i++) {
    for (let j = 0; j < a[0].length; j++) {
      if (a[i][j] === 1 && res[i][j] < 4) {
        sum += 1
      }
    }
  }
  return sum
}
part1(input)

1395

Great! the correct answer. The second part now asks, what if remove the eligible ones and repeat the process until we converge. How many could we get? If we generate the new array then we can just repeat this until we stop.

In [17]:
let inter = map2(a, x => x)

In [18]:
res = conv2d(inter, ker)
let sum = 0; 
for (let i = 0; i < a.length; i++) {
  for (let j = 0; j < a[0].length; j++) {
    if (inter[i][j] === 1 && res[i][j] < 4) {
      sum += 1
      inter[i][j] = 0 // usually a bad idea to mutate something we iterate on but in this case its okay
    }
  }
}
console.log(sum)
pp(inter)

13
0 0 0 0 0 0 0 1 0 0
0 1 1 0 1 0 1 0 1 1
1 1 1 1 1 0 0 0 1 1
1 0 1 1 1 1 0 0 1 0
0 1 0 1 1 1 1 0 1 0
0 1 1 1 1 1 1 1 0 1
0 1 0 1 0 1 0 1 1 1
0 0 1 1 1 0 1 1 1 1
0 1 1 1 1 1 1 1 1 0
0 0 0 0 1 1 1 0 0 0


So we got back the sum but also a new version of the array with some of the ones removed. let's put it in a function and iterate

In [19]:
function step(a, ker) {
  let inter = map2(a, x => x)
  const res = conv2d(inter, ker)
  let sum = 0
  for (let i = 0; i < a.length; i++) {
    for (let j = 0; j < a[0].length; j++) {
      if (inter[i][j] === 1 && res[i][j] < 4) {
        sum += 1
        inter[i][j] = 0
      }
    }
  }
  return [sum, inter]
}

In [20]:
let [s, inter] = step(a, ker)
console.log(s)
pp(inter)

13
0 0 0 0 0 0 0 1 0 0
0 1 1 0 1 0 1 0 1 1
1 1 1 1 1 0 0 0 1 1
1 0 1 1 1 1 0 0 1 0
0 1 0 1 1 1 1 0 1 0
0 1 1 1 1 1 1 1 0 1
0 1 0 1 0 1 0 1 1 1
0 0 1 1 1 0 1 1 1 1
0 1 1 1 1 1 1 1 1 0
0 0 0 0 1 1 1 0 0 0


In [21]:
[s, inter] = step(inter, ker)
console.log(s)
pp(inter)

12
0 0 0 0 0 0 0 0 0 0
0 1 1 0 0 0 0 0 1 0
0 1 1 1 1 0 0 0 1 1
0 0 1 1 1 1 0 0 0 0
0 1 0 1 1 1 1 0 0 0
0 0 1 1 1 1 1 1 0 0
0 0 0 1 0 1 0 1 1 1
0 0 1 1 1 0 1 1 1 1
0 0 1 1 1 1 1 1 1 0
0 0 0 0 1 1 1 0 0 0


In [22]:
[s, inter] = step(inter, ker)
console.log(s)
pp(inter)

7
0 0 0 0 0 0 0 0 0 0
0 0 1 0 0 0 0 0 0 0
0 1 1 1 1 0 0 0 0 0
0 0 1 1 1 1 0 0 0 0
0 0 0 1 1 1 1 0 0 0
0 0 1 1 1 1 1 1 0 0
0 0 0 1 0 1 0 1 1 0
0 0 1 1 1 0 1 1 1 1
0 0 0 1 1 1 1 1 1 0
0 0 0 0 1 1 1 0 0 0


In [23]:
[s, inter] = step(inter, ker)
console.log(s)
pp(inter)

5
0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0
0 0 1 1 1 0 0 0 0 0
0 0 1 1 1 1 0 0 0 0
0 0 0 1 1 1 1 0 0 0
0 0 0 1 1 1 1 1 0 0
0 0 0 1 0 1 0 1 1 0
0 0 0 1 1 0 1 1 1 0
0 0 0 1 1 1 1 1 1 0
0 0 0 0 1 1 1 0 0 0


In [24]:
[s, inter] = step(inter, ker)
console.log(s)
pp(inter)

2
0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0
0 0 0 1 1 0 0 0 0 0
0 0 1 1 1 1 0 0 0 0
0 0 0 1 1 1 1 0 0 0
0 0 0 1 1 1 1 1 0 0
0 0 0 1 0 1 0 1 1 0
0 0 0 1 1 0 1 1 1 0
0 0 0 1 1 1 1 1 0 0
0 0 0 0 1 1 1 0 0 0


And so on. Let's create the final function.

In [43]:
function part2(input) {
  const a = process(input)
  const ker = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]
  let res = 0
  let [s, inter] = step(a, ker)
  while (s > 0) {
    res += s;
    [s, inter] = step(inter, ker)
  }
  return res
}

In [44]:
part2(input)

8451

Correct again!